In [ ]:
pip install contractions

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [ ]:
import pandas as pd
import re 
import string 
import contractions

import nltk
nltk.download('stopwords')
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer
nltk.download('punkt_tab', quiet=True)

nltk.download('punkt')
nltk.download('wordnet')
nltk.download('omw-1.4')

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\Rajguru\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\Rajguru\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\Rajguru\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     C:\Users\Rajguru\AppData\Roaming\nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


True

In [ ]:
df = pd.read_csv("data/IMDB Dataset.csv")
df.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


In [ ]:
stop_word = set(stopwords.words('english'))
lemmetizer = WordNetLemmatizer()

In [ ]:
def preprocess_text(text):
    #Lowercace
    text = text.lower()
    #Expand contractions
    text = contractions.fix(text)
    #Remove THML tags
    text = re.sub(r'<.?>','',text)
    #Remove URLs
    text = re.sub(r'httls?://\S+|www\S+','',text)
    #Remove Email ids 
    text = re.sub(r'\S+@\S+','',text)

    #Remove Numbers
    text = re.sub(r'\d+','',text)

    #Remove punctuation
    text = text.translate(str.maketrans('', '', string.punctuation))

    #Remove Repeated Characters
    text = re.sub(r'(.)\1{2,}',r'\1',text)

    #Tokenization
    tokens = word_tokenize(text)

    #Remove Stopwords
    filtered_tokens = []

    for words in tokens:
        if words not in stop_word:
            filtered_tokens.append(words)

    tokens = filtered_tokens

    #Lemmatization
    lemmatized_tokens = []

    for words in tokens:
        lemma = lemmetizer.lemmatize(words)
        lemmatized_tokens.append(lemma)

    tokens = lemmatized_tokens

    #Join tokens back in sentence
    text = " ".join(tokens)

    return text 




In [7]:
df["clean_review"] = df["review"].apply(preprocess_text)

In [8]:
df[['review','clean_review']].head()

,review,clean_review
0,One of the other reviewers has mentioned that ...,one reviewer mentioned watching oz episode hoo...
1,A wonderful little production. <br /><br />The...,wonderful little production br br filming tech...
2,I thought this was a wonderful way to spend ti...,thought wonderful way spend time hot summer we...
3,Basically there's a family where a little boy ...,basically family little boy jake think zombie ...
4,"Petter Mattei's ""Love in the Time of Money"" is...",petter matteis love time money visually stunni...


In [9]:
df.to_csv("./data/IMDB_Cleaned.csv", index=False)

In [10]:
from sklearn.feature_extraction.text import CountVectorizer
import pandas as pd 

documents ={
    "I love this movie",
    "This movie is fantastic",
    "I hate this movie",
    "Thin file is boring",
    "I love this fnatastic film"
}
vectorize =CountVectorizer()
bow = vectorize.fit_transform(documents)

In [11]:
bow

<Compressed Sparse Row sparse matrix of dtype 'int64'
	with 18 stored elements and shape (5, 11)>

In [12]:
bow.toarray()

array([[0, 0, 0, 0, 0, 0, 0, 1, 1, 0, 1],
       [0, 1, 0, 0, 0, 0, 1, 0, 1, 0, 1],
       [0, 0, 0, 1, 1, 0, 0, 1, 0, 0, 1],
       [1, 0, 1, 0, 0, 0, 1, 0, 0, 1, 0],
       [0, 0, 0, 0, 0, 1, 0, 0, 1, 0, 1]])

In [13]:
vectorize.get_feature_names_out()

array(['boring', 'fantastic', 'file', 'film', 'fnatastic', 'hate', 'is',
       'love', 'movie', 'thin', 'this'], dtype=object)

In [14]:
df = pd.DataFrame(
    bow.toarray(),
    columns=vectorize.get_feature_names_out()
)
df.head()

,boring,fantastic,file,film,fnatastic,hate,is,love,movie,thin,this
0,0,0,0,0,0,0,0,1,1,0,1
1,0,1,0,0,0,0,1,0,1,0,1
2,0,0,0,1,1,0,0,1,0,0,1
3,1,0,1,0,0,0,1,0,0,1,0
4,0,0,0,0,0,1,0,0,1,0,1


In [15]:
import sklearn as sk
from sklearn.feature_extraction.text import TfidfVectorizer 
import pandas as pd 


In [26]:
tfidf = TfidfVectorizer()

X_tfidf = tfidf.fit_transform(df["clean_review"])

X_tfidf

KeyError: 'clean_review'

In [22]:
tfidf = TfidfVectorizer(max_features=5000)

In [24]:
X_tfidf = tfidf.fit_transform(df["clean_review"])
X_tfidf

AttributeError: 'csr_matrix' object has no attribute 'fit_transform'

In [17]:
tfidf

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 18 stored elements and shape (5, 11)>

In [20]:
tfidf.toarray()

array([[0.        , 0.        , 0.        , 0.        , 0.        ,
        0.        , 0.        , 0.6778033 , 0.56263796, 0.        ,
        0.47330879],
       [0.        , 0.64324583, 0.        , 0.        , 0.        ,
        0.        , 0.51896668, 0.        , 0.43078923, 0.        ,
        0.36239348],
       [0.        , 0.        , 0.        , 0.58042343, 0.58042343,
        0.        , 0.        , 0.46828197, 0.        , 0.        ,
        0.32700044],
       [0.52335825, 0.        , 0.52335825, 0.        , 0.        ,
        0.        , 0.42224214, 0.        , 0.        , 0.52335825,
        0.        ],
       [0.        , 0.        , 0.        , 0.        , 0.        ,
        0.75251519, 0.        , 0.        , 0.5039682 , 0.        ,
        0.42395393]])

In [21]:
tfidf.get_feature_names_out()



AttributeError: 'csr_matrix' object has no attribute 'get_feature_names_out'

In [ ]:
tfidf_df = pd.DataFrame(
    X_tfidf.toarray()
    column=tfidf.get
)